# Emission-line fitting and abundances with `jwspecfit`: an example

From a calibrated NIRSpec spectrum to line fluxes, equivalent widths and chemical abundances, using SPURS 437 (z = 5.77, G140M + G235M + G395M combined), shipped with the package in `data/`.

**More information**
- Paper (method and validation): [Rai & Roberts-Borsani (2026), arXiv:2608.10063](https://arxiv.org/abs/2608.10063). **Please cite it if you use the code.**
- Code and issues: [github.com/raunaq-rai/jwspecfit](https://github.com/raunaq-rai/jwspecfit)
- Documentation and API reference: [jwspecfit.readthedocs.io](https://jwspecfit.readthedocs.io/)

## 1. What it does

- **Line fits.** Each line is a Gaussian integrated over each pixel, at the spectrum's own resolution (read from the file), on top of a continuum. Broad components (Balmer, [O III], He I) can be added and are kept only if the BIC prefers them.
- **Continuum.** By default a polynomial of degree `deg` (3 here), fitted with the regions around known lines masked and positive outliers rejected iteratively (`clip_sigma`, default 2.5σ), then subtracted before the lines are fitted. Alternatives: `moving_average=True` for a running median (75 pixels, or give the window as an integer), or, in `jwspecmcmc`, your own continuum with `continuum=` (e.g. an SED fit).
- **Measurements.** Flux, error, S/N, equivalent width (rest frame and observed frame), centroid and width for every line, and 3σ upper limits on the flux and rest-frame EW of any line that is not detected.
- **Abundances.** Dust correction from the Balmer decrement, with the Salim et al. (2018) attenuation curve by default (`dust_law="cardelli"` for Cardelli et al. 1989, or set A_V yourself with `Av=`); direct-method O/H from [O III] 4363 or O III] 1666, including the joint T_e-n_e solve; strong-line O/H when the auroral lines are too faint; C/O and N/O with ionisation correction factors; upper limits for ions that are not detected.

**Which fitter.** I recommend `jwspecmcmc` (MCMC with the NUTS sampler) for any measurement you will publish:
- priors are easy to set (`prior_overrides`), and new features go into it first;
- convergence can be checked (R̂ and ESS, section 6);
- it runs in JAX, so it is fast: in this notebook it beats the bootstrap fit while drawing ~80 times as many samples;
- every line has a full posterior, which the abundance code propagates directly.

The least-squares fitter with bootstrap errors (`jwspecfit`) is still what starts the MCMC chains. It is at the end of the notebook as an extra (section 9).

**What is fitted and reported**
- **Fitted:** every line in `LINES`, detected or not. Only lines the spectrum does not cover are dropped (section 5).
- **Reported:** every fitted line has a flux, error, S/N and EW. Decide detection from `snr` (S/N ≥ 3 is typical). For a line that is not detected, quote `upper_limit(3)`, never its fitted flux.
- **S/N:** the line's fitted flux (the integral of its Gaussian) divided by the 1σ error on that flux; it is not a sum over pixels. In `jwspecmcmc` the error is the mean of the posterior's lower and upper 1σ (16th to 84th percentiles); in `jwspecfit` it is the bootstrap error. So S/N 3 means the flux is three times its own uncertainty.
- **Upper limit:** for a line that is not detected, quote `upper_limit(3)`. It is built from the noise in the spectrum, never from the line's fitted flux or width (both meaningless for an undetected line):
  1. **Profile.** A Gaussian at the line's expected wavelength, with the width a real line would have: σ² = σ_inst² + σ_v², where σ_inst comes from the resolution at that wavelength and σ_v is the galaxy's velocity width, measured from its detected lines.
  2. **Noise from the error spectrum.** The flux error of that Gaussian, σ_F = √(Σ gᵢ² eᵢ²) / Σ gᵢ², summed over the line's pixels i, with g the profile and e the error spectrum.
  3. **Correction from the residuals.** σ_F is multiplied by a factor f measured from the residuals (data − model) around the line, out to ±50σ either side, the line itself excluded (the larger of the two sides is kept). f = 1 if the error spectrum is right; f > 1 corrects for correlated pixels or errors that are too small (1.03 to 1.15 is typical for the gratings).
  4. **Limit.** `upper_limit(3)` = 3 × f × σ_F.
  5. **EW limit.** `ew0_upper_limit(3)` = `upper_limit(3)` / continuum at the line (`cont_flam`, the fitted continuum interpolated under the line) / (1 + z), so a line's EW and flux limits always agree.

  Both fitters and the abundance limits (section 7) use this. It assumes one line: for a blended entry such as `OII_doublet` the flux is spread over two lines, so the limit comes out too low; fit `OII_3726` and `OII_3729` separately instead.

## 2. Installation

`jwspecmcmc` and `jwspecabund` come with `jwspecfit`: one install gives all three. The MCMC samplers and the abundance code need optional extras, which the commands below include.

**1. A fresh environment** (Python 3.11 or 3.12: JAX, behind the NUTS sampler, needs 3.11 or newer)

```bash
conda create -n jwspecfit python=3.12
conda activate jwspecfit
```

**2. jwspecfit with every extra, plus what this notebook uses**

```bash
pip install "jwspecfit[nuts,mcmc,abund]>=1.1.14"
pip install jupyterlab ipykernel ipywidgets pandas nbformat
python -m ipykernel install --user --name jwspecfit --display-name "Python (jwspecfit)"
```

**From source**, to follow the latest version or edit the code:

```bash
git clone https://github.com/raunaq-rai/jwspecfit.git
cd jwspecfit
pip install -e ".[nuts,mcmc,abund]"
```

In [ ]:
import importlib.metadata as md_

for pkg in ("jwspecfit", "jax", "numpyro", "pyneb", "emcee", "plotly", "pandas", "ipywidgets"):
    try:
        print(f"{pkg:10s} {md_.version(pkg)}")
    except md_.PackageNotFoundError:
        print(f"{pkg:10s} MISSING: see the install commands above")

## 3. Imports

**Set the number of chains first.** NumPyro must be told how many CPU devices it may use *before* JAX is imported. Without it, NumPyro sees one device and the chains run one after another, which is slow and can stop NUTS from working properly. Set `N_CHAINS` to the number of chains you want (at most your number of CPU cores); the same number is used for the MCMC fit below.

In [ ]:
N_CHAINS = 4  # MCMC chains, one CPU core each; must be set before jax / numpyro are imported

import numpyro

numpyro.set_host_device_count(N_CHAINS)

import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from astropy.io import fits

import jwspecabund
import jwspecfit
import jwspecmcmc

logging.getLogger("jwspecabund").setLevel(logging.ERROR)  # keep the abundance output to the summary
print(f"jwspecfit {jwspecfit.__version__}, jwspecmcmc {jwspecmcmc.__version__}, jwspecabund {jwspecabund.__version__}")

import jax

print(f"JAX sees {jax.local_device_count()} CPU devices for {N_CHAINS} chains")  # must match; else restart the kernel

## 4. The spectrum: SPURS 437

The combined file holds the three gratings joined into one spectrum (`SPEC1D`); the redshift is in the header (`ZSPEC`). The file's `resolution` column (the R of this source, not the nominal grating value) is read too and used for every instrumental width. The interactive plot lets you zoom in and toggle the line markers.

In [ ]:
from pathlib import Path

PATH = Path("../../data/spurs437_a2744_combined-medium_spec1d.fits")  # relative to docs/notebooks/

hdr = fits.getheader(PATH)
z = hdr["ZSPEC"]
print(f"SPURS {hdr['SPURS_ID']}: z = {z:.4f} ({hdr['ZQUAL']})")

spec = jwspecfit.read_fits(PATH, z=z)
jwspecfit.plot_spectrum_interactive(spec, z=z, rest_frame=True, title=f"SPURS 437, z = {z:.3f}")

## 5. The lines to fit

Lines are given by name. To see every built-in line, with its rest wavelength:

```python
jwspecfit.show_lines()                # prints the full list; show_lines(search="OIII") to filter
```

- **Default (`lines=None`):** every line in the default list that the spectrum covers is fitted (the resolved-line list for gratings, a shorter one for the prism).
- **Your own list:** set `LINES` as below and pass `lines=LINES`.
- **A line that is not in the list:** add it to `jwspecfit.REST_LINES_A` (name → rest-frame **vacuum** wavelength in Å), then add its name to `LINES`:

```python
jwspecfit.REST_LINES_A["MY_LINE"] = 9999.9   # rest-frame vacuum wavelength, Angstrom
LINES.append("MY_LINE")
```

  A new line is fitted on its own (free flux, centroid and width) and gets a flux, S/N and upper limit like any other, but the abundance code does not use it. Check `show_lines()` first: many lines are already there. The entry lasts only until the kernel restarts.

Lines the spectrum does not cover (fewer than half their pixels valid: outside the wavelength range, or in a detector gap) are dropped. Here N IV] and C IV, below 1.054 µm at z = 5.77, are dropped. Listing lines you expect but do not see is still useful: they get an upper limit.

In [ ]:
# run the below to show the lines you can fit, or add your own using the instructions above

jwspecfit.show_lines()

In [ ]:
LINES = [
    # rest UV
    "NIV_1483", "NIV_1486", "CIV_1", "CIV_2", "HEII_1640", "OIII_1661", "OIII_1666", "NIII_1750",
    "CIII]_1907", "CIII]_1909",
    # rest optical
    "OII_doublet", "NeIII_3869", "HDELTA", "HGAMMA", "OIII_4363", "HBETA", "OIII_4959", "OIII_5007",
    "HALPHA", "NII_6585",
]

## 6. Line fitting: `jwspecmcmc` (NUTS)


| Setting | Use | Why |
|---|---|---|
| `numpyro.set_host_device_count(N_CHAINS)` | before importing jax (section 3) | otherwise the chains run one after another and NUTS can fail |
| `n_chains=N_CHAINS` | 4 or more | R̂ compares chains, so it needs at least 2; more chains give a more reliable R̂ |
| resolution | `spec.R`, from the file | read by `read_fits` (section 4); sets the instrumental line widths |
| `init_from_mle=True` | default, keep it | starts every chain from a quick least-squares fit, so none starts out of bounds |
| `n_warmup`, `n_samples_nuts` | 500 and 2000 per chain | raise `n_warmup` if R̂ comes out above 1.01 |
| `target_accept_prob` | 0.8 by default; up to 0.95 | raise it if the sampler reports divergences |
| `tie_balmer_to_oiii=True` | default | Balmer lines share [O III]'s velocity and width, so faint Hδ and Hγ are measured at the right width |
| `tie_uv_*`, `fix_blended_ratios` | as below | how UV doublets and blends are tied |

Each chain runs on its own CPU core.

In [ ]:
mcmc = jwspecmcmc.fit_lines(
    spec, z=z,
    sampler="nuts",
    lines=LINES,
    deg=3,
    n_chains=N_CHAINS,
    n_warmup=500,
    n_samples_nuts=2000,
    tie_balmer_to_oiii=True,  # tie the Balmer lines to OIII 5007 for the redshift and width
    tie_uv_doublets=True, tie_uv_centroids=True, tie_uv_widths=False,
    fix_blended_ratios=False,
    seed=2409,
)

### Convergence

Each parameter gets the rank-normalised split R̂ (Vehtari et al. 2021) and an effective sample size (ESS). R̂ compares the spread within each chain with the spread between chains: **R̂ < 1.01** means the chains agree. Ask also for **ESS > 100**; `converged` is `True` only when both hold for every parameter.

In [ ]:
conv = mcmc.convergence
print(f"converged: {conv['converged']}   worst R-hat: {conv['r_hat_max']:.4f}   lowest ESS: {conv['ess_min']:.0f}")

### The fit

`to_fit_result()` turns the posterior medians into the same kind of result as the least-squares fit, so the same plotting functions work.

In [ ]:
jwspecfit.plot_fit_interactive(mcmc.to_fit_result(), rest_frame=True)

### Accessing the measurements

`mcmc.lines` holds one `MCMCLineResult` per line. Values are posterior medians; errors are the 16th to 84th percentiles, given as `(lower, upper)`.

| Attribute | Meaning |
|---|---|
| `flux`, `flux_err` | median flux and its `(lower, upper)` 1σ errors (erg s⁻¹ cm⁻²) |
| `flux_posterior` | every posterior sample of the flux |
| `ew0_A`, `ew0_err`, `ew0_posterior` | rest-frame equivalent width (Å), its errors and samples (flux and continuum uncertainty both included) |
| `ew_obs_A`, `ew_obs_err` | observed-frame equivalent width, `ew0_A` × (1 + z) |
| `cont_flam`, `cont_err_flam`, `cont_snr` | the continuum at the line that the EW is measured against (f_λ), its error and S/N |
| `snr` | flux / mean error |
| `centroid_A`, `sigma_A` (and `_err`) | observed centroid and σ |
| `upper_limit(n_sigma=3)`, `ew0_upper_limit(n_sigma=3)` | 3σ flux and rest-frame EW limits for a non-detection (section 1) |

and on the result itself: `flux_ratio_posterior(a, b)` for any ratio, `flat_chains` and `chains` for the raw samples.

In [ ]:
o3 = mcmc.lines["OIII_5007"]
lo, hi = o3.flux_err
print(f"[O III] 5007: {o3.flux:.3e} (-{lo:.1e} / +{hi:.1e}) erg/s/cm2, S/N {o3.snr:.0f}, EW0 {o3.ew0_A:.0f} A (observed {o3.ew_obs_A:.0f} A)")

table_mcmc = pd.DataFrame([
    {"line": n, "flux": lr.flux, "err_lo": lr.flux_err[0], "err_hi": lr.flux_err[1], "snr": lr.snr,
     "ew0_A": lr.ew0_A, "ew_obs_A": lr.ew_obs_A,
     "upper_limit_3sig": lr.upper_limit(3), "ew0_upper_limit_3sig": lr.ew0_upper_limit(3)}
    for n, lr in mcmc.lines.items()
]).set_index("line")
table_mcmc

## 7. Abundances: `jwspecabund`

`compute_abundances` takes the MCMC result and resamples its flux posteriors, so every abundance has a posterior too. The steps, in order: dust from the Balmer decrement, electron densities, electron temperatures, ionic abundances, then O/H, C/O and N/O.

| Setting | Here | Meaning |
|---|---|---|
| `method` | `"auto"` | direct (T_e) method if an auroral line is detected, otherwise strong-line; or force `"direct"`, `"strong_line"`, `"forward"` |
| `self_consistent_OIII` | `"auto"` | solve T_e and n_e together from O III] 1666, [O III] 4363 and 5007 when all three pass `snr_auroral` (section 8) |
| `snr_auroral` | 2 | S/N an auroral line needs for the direct method and the joint solve (default 3). [O III] 4363 in SPURS 437 is only a 2.5σ to 3.4σ line (MCMC and bootstrap), so it is lowered here to show the joint solve: treat the result as tentative, and keep 3 unless you have a reason |
| `snr_line` | 2 | S/N below which a line is treated as undetected (and gets an upper limit) |
| `dust_law` | `"salim"` | the attenuation curve: Salim et al. (2018), the default, or `"cardelli"` for Cardelli et al. (1989). Or give `Av=` to fix A_V instead of measuring it |
| `balmer_anchor` | `"HALPHA"` | decrements against Hα (Hβ/Hα, Hγ/Hα, ...); `"HBETA"` to leave Hα out |
| `icf_method` | `"auto"` | N/O correction: Martinez+25 when log U is constrained; or `"izotov06"`, `"direct_sum"` |
| `n_posterior` | 100 here | posterior draws propagated; use 1000 or more for results |

In [ ]:
abund = jwspecabund.compute_abundances(
    mcmc, z=z,
    method="auto",
    self_consistent_OIII="auto",  # the joint T_e-n_e solve (section 8)
    snr_line=3.0, #The integrated SNR threshold above which a line is considered detected. Calculated by integrating the data (flux/error) over the best Gaussian fit.
    snr_auroral=2.0,              # default 3; [O III] 4363 is only ~2.5 sigma in this galaxy - the auroral line has its own snr argument because it is used to compute T_e and n_e, and the auroral line is often weak.
    dust_law="salim",
    balmer_anchor="HALPHA",
    icf_method="auto",
    n_posterior=100, # do >1000 
    progress=True,
)
print(abund.summary())

### Accessing the results

**What was done.** `method` is the metallicity method used, `Te_diagnostic` the temperature diagnostic, and `diagnostics` a dictionary with one sentence per step (which lines, which density, which relation). `failures` says why anything could not be computed, `excluded_lines` lists the lines dropped below `snr_line`, and `alt_results` holds the methods that were possible but not adopted.

In [ ]:
print(f"metallicity method : {abund.method}")
print(f"T_e diagnostic     : {abund.Te_diagnostic}")
print(f"N/O ICF            : {abund.icf_method} ({abund.NO_icf_name})")
print(f"excluded lines     : {abund.excluded_lines}")
print()
for step, what in (abund.diagnostics or {}).items():
    print(f"{step}: {what}\n")
print("could not compute:")
for quantity, why in (abund.failures or {}).items():
    print(f"  {quantity}: {why}")

**The numbers.** Errors are 1σ, either one number or `(lower, upper)`; every `*_posterior` holds the samples.

| Attribute | Meaning |
|---|---|
| `OH`, `OH_err`, `OH_posterior` | 12 + log(O/H) |
| `Te_high`, `Te_mid`, `Te_low` (and `_err`) | T_e of the high-, intermediate- and low-ionisation zones (K) |
| `ne_low`, `ne_mid`, `ne_high`, `ne_Opp` | electron densities (cm⁻³); `ne_Opp` from the joint solve |
| `Av`, `Av_err` | V-band attenuation from the Balmer decrement |
| `CO`, `NO`, `NeO`, `SO`, `ArO` (and `_err`, `_posterior`) | log abundance ratios (`None` if not computed) |
| `logU` | ionisation parameter, when constrained |
| `ionic` | every ionic abundance X^i+/H+ (0 for an undetected ion) |
| `ionic_upper_limits`, `ionic_ul_details` | 3σ limits on the undetected ions, and the lines and flux limits behind them |

In [ ]:
def pm(value, err):
    # value with symmetric or (lower, upper) errors
    if value is None:
        return "not computed"
    if isinstance(err, (tuple, list)):
        return f"{value:.3f} (-{err[0]:.3f} / +{err[1]:.3f})"
    return f"{value:.3f} +/- {err:.3f}" if err is not None else f"{value:.3f}"


print(f"12 + log(O/H) = {pm(abund.OH, abund.OH_err)}")
print(f"T_e(high)     = {pm(abund.Te_high, abund.Te_high_err)} K")
print(f"A_V           = {pm(abund.Av, abund.Av_err)}")
print(f"log(C/O)      = {pm(abund.CO, abund.CO_err)}")
print(f"log(N/O)      = {pm(abund.NO, abund.NO_err)}")
print("\nionic abundances:")
for ion, value in (abund.ionic or {}).items():
    print(f"  {ion:8s} {value:.3e}" if value > 0 else f"  {ion:8s} not detected")

**Upper limits.** When every line of an ion is below `snr_line`, the ion has no abundance (0 in `ionic`) and gets a 3σ upper limit instead (`ionic_upper_limits`):

1. Take the 3σ flux limits of its lines, `upper_limit(3)` (section 1). A doublet's two limits are added in quadrature; the N III] quintuplet, fitted as one blend, uses its largest member.
2. Dust-correct with the same A_V as the detected lines.
3. Convert to an abundance at that zone's T_e and n_e.

`ionic_ul_details[ion]` lists the lines and the flux limit used. A ratio with an undetected ion becomes a limit too: for N/O, (detected N ions + limits on the undetected ones) / (O⁺ + O²⁺).

In [ ]:
for ion, limit in (abund.ionic_upper_limits or {}).items():
    d = abund.ionic_ul_details[ion]
    print(f"{ion:8s} < {limit:.3e}   from {', '.join(d['lines'])}: flux < {d['flux_ul']:.2e} "
          f"({d['n_sigma']:.0f} sigma, {d['method']})")

# N/O upper limit, as in the summary: detected N ions + limits on the undetected ones, over O+ + O++
ionic, uls = abund.ionic or {}, abund.ionic_upper_limits or {}
O_total = ionic.get("O+/H+", 0.0) + ionic.get("O++/H+", 0.0)
N_total, any_limit = 0.0, False
for ion in ("N+/H+", "N++/H+", "N+++/H+"):
    if ionic.get(ion, 0.0) > 0:
        N_total += ionic[ion]
    elif ion in uls:
        N_total += uls[ion]
        any_limit = True
if abund.NO is None and any_limit and O_total > 0:
    print(f"\nlog(N/O) < {np.log10(N_total / O_total):.3f}")

**Alternatives.** Each entry of `alt_results` is a full result for a method that could be used but was not adopted, for comparison.

In [ ]:
for name, alt in (abund.alt_results or {}).items():
    line = f"{name:12s}"
    if alt.OH is not None and np.isfinite(alt.OH):
        line += f" 12+log(O/H) = {alt.OH:.3f}"
    if alt.Te_high:
        line += f"   T_e = {alt.Te_high:.0f} K"
    if alt.CO is not None:
        line += f"   log(C/O) = {alt.CO:.3f}"
    print(line)

## 8. The joint T_e-n_e solve

[O III] 5007 has a critical density of only ~7 × 10⁵ cm⁻³, so in the dense gas common at high redshift it is collisionally de-excited and the classical [O III] 4363/5007 ratio depends on the density as well as the temperature. Solving it at an assumed low density overestimates T_e and underestimates O/H. O III] 1666 has a much higher critical density, so with all three lines there are two independent ratios for the two unknowns, and T_e and n_e are solved together (Hsiao et al. 2026).

Each ratio traces a curve of (T_e, n_e) pairs; where the curves cross, one pair reproduces both, and that is the solution. Curves that stay parallel mean the density is not constrained, and only an upper limit on n_e follows.

With `self_consistent_OIII="auto"` and all three lines above `snr_auroral`, `compute_abundances` runs the joint solve, keeps it in `Te_ne_selfconsistent` and uses it for O/H (`Te_diagnostic` is then `"self_consistent"`). It is kept also when the curves only bound n_e from above; n_e is then quoted as an upper limit.

In [ ]:
sc = abund.Te_ne_selfconsistent
if sc is None:
    print("No joint solve: O III] 1666, [O III] 4363 and 5007 are not all detected")
else:
    print(f"T_e(O++) = {sc.Te:.0f} (-{sc.Te_err[0]:.0f} / +{sc.Te_err[1]:.0f}) K")
    if sc.ne_is_upper_limit:
        print(f"n_e(O++) < {sc.ne_upper_limit:.3g} cm^-3 (curves parallel: upper limit only)")
    else:
        print(f"n_e(O++) = {sc.ne:.3g} (-{sc.ne_err[0]:.2g} / +{sc.ne_err[1]:.2g}) cm^-3")
    print(f"curves intersect: {sc.converged}   ({100 * sc.converged_fraction:.0f}% of posterior draws)")
    print("\nfor comparison, solving one ratio at a borrowed density:")
    for name in ("direct_4363", "direct_1666"):
        alt = (abund.alt_results or {}).get(name)
        if alt is not None:
            print(f"  {name}: T_e = {alt.Te_high:.0f} K, 12+log(O/H) = {alt.OH:.3f}")

The three T_e(n_e) curves with their 1σ bands, the posterior (1σ and 2σ contours) and the adopted solution (star).

In [ ]:
if abund.Te_ne_selfconsistent is not None:
    fig = jwspecabund.plot_te_ne_diagnostic(abund)
    plt.show()

The star is where the curves from the measured ratios cross (`self_consistent_point="intersection"`, the default). The shaded contours are the posterior of the draws; `self_consistent_point="median"` reports their medians instead, which need not lie on the crossing.

## 9. Extra: least-squares fit with `jwspecfit` (bootstrap errors)

The same model fitted by least squares, with errors from `n_boot` bootstrap resamples of the spectrum (100 to 200 for a look, 1000 for numbers you keep). It has no priors and no convergence check, so I use it for quick looks; it is also what starts the MCMC chains.

In [ ]:
fit = jwspecfit.fit_lines(
    spec, z=z,
    lines=LINES,
    deg=3,                   # continuum polynomial degree
    n_boot=100,              # bootstrap resamples for the errors
    tie_uv_doublets=True,    # UV doublets share kinematics
    tie_uv_centroids=True,   # UV lines share one velocity
    tie_uv_widths=False,     # but keep their own widths
    fix_blended_ratios=False,  # keep both amplitudes of a blended doublet free
)
print(f"chi2/dof = {fit.chi2:.2f}")

In [ ]:
jwspecfit.plot_fit_interactive(fit, rest_frame=True)

### Accessing the measurements

`fit.lines` is a dictionary of `LineResult`s, one per line. Fluxes are in erg s⁻¹ cm⁻² (observed, not dust- or lensing-corrected), equivalent widths in Å (`ew0_A` rest frame, `ew_obs_A` observed), centroids and widths in observed Å.

| Attribute | Meaning |
|---|---|
| `flux`, `flux_err` | integrated flux and its bootstrap 1σ error |
| `snr` | flux / flux_err (the S/N used throughout) |
| `snr_int_cont` | flux / (local continuum RMS × √(pixels within ±3σ)), a noise-based alternative |
| `snr_peak_err`, `snr_peak_cont` | peak of the fitted Gaussian over the error array, or over the local continuum RMS, at the line centre |
| `ew0_A`, `ew_obs_A` | rest-frame and observed-frame equivalent width (Å) |
| `cont_flam` | the continuum at the line that the EW is measured against (f_λ) |
| `centroid_A`, `sigma_A` | observed centroid and Gaussian σ (Å, including the instrument) |
| `upper_limit(n_sigma=3)`, `ew0_upper_limit(n_sigma=3)` | 3σ flux and rest-frame EW limits for a non-detection (section 1) |

In [ ]:
o3 = fit.lines["OIII_5007"]
print(f"[O III] 5007: {o3.flux:.3e} +/- {o3.flux_err:.1e} erg/s/cm2, S/N {o3.snr:.0f}, EW0 {o3.ew0_A:.0f} A (observed {o3.ew_obs_A:.0f} A)")
n3 = fit.lines["NIII_1750"]
print(f"N III] 1750 3-sigma upper limits: flux {n3.upper_limit(3):.2e} erg/s/cm2, EW0 {n3.ew0_upper_limit(3):.1f} A")

table = pd.DataFrame([
    {"line": n, "flux": lr.flux, "flux_err": lr.flux_err, "snr": lr.snr, "ew0_A": lr.ew0_A, "ew_obs_A": lr.ew_obs_A,
     "upper_limit_3sig": lr.upper_limit(3), "ew0_upper_limit_3sig": lr.ew0_upper_limit(3)}
    for n, lr in fit.lines.items()
]).set_index("line")
table

In [ ]:
jwspecfit.export_lines_txt(fit, "spurs437_jwspecfit_lines.txt")  # a plain-text line table
jwspecfit.save_result(fit, "spurs437_jwspecfit.npz")             # the whole fit, to reload with load_result